# battery_scenario_validation.ipynb (Sarvesh) - battery scenarios + contingencies
Extends the existing scenario library (keep your existing 9x25 table if already used) with battery-state scenarios and **real** contingencies (line outage, transformer derating, switch opening, generator loss). Every scenario changes real loads / generation / storage / topology **before** pandapower is re-run.

In [ ]:
import sys, io, warnings
from contextlib import redirect_stdout, redirect_stderr

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import simbench as sb
import pandapower as pp

sys.path.insert(0, ".")                       # WHY: import sarvesh_lib.py from digital_twin/
from sarvesh_lib import *

net = sb.get_simbench_net("1-MVLV-urban-all-0-sw")

print(len(net.bus), "buses |", len(net.storage),
      "storage units (should be 0 before we add ours)")


In [ ]:
import json

cfg = BatteryConfig(**json.load(open("output/battery_config.json")))
sidx = add_battery(net, cfg)
snap = snapshot_state(net)

N_SAMPLES = 25
table = build_scenario_table(n=N_SAMPLES, seed=0)
table.to_csv("output/scenario_library.csv", index=False)

# Only cache the operating points actually used by the 300 scenarios plus annual peak.
needed_steps = sorted(set(table["profile_step"].astype(int).tolist()) | {33001})

# Slice the small relative-profile tables BEFORE converting to absolute values.
load_rel = net.profiles["load"].iloc[needed_steps].copy()
pp_rel = net.profiles["powerplants"].iloc[needed_steps].copy()
res_rel = net.profiles["renewables"].iloc[needed_steps].copy()

# Merge generator profile tables without expanding the full year.
assert pp_rel.index.equals(res_rel.index)
time_col = res_rel["time"].copy() if "time" in res_rel.columns else None

pp_data = pp_rel.drop(columns=["time"], errors="ignore")
res_data = res_rel.drop(columns=["time"], errors="ignore")
sgen_rel = pd.concat([pp_data, res_data], axis=1)

if time_col is not None:
    sgen_rel["time"] = time_col

assert not sgen_rel.columns.duplicated().any(), "Duplicate sgen profile names found"

# IMPORTANT: SimBench load P and Q profiles require their proper suffixes.
profiles = {
    ("load", "p_mw"): sb.get_absolute_profiles_from_relative_profiles(
        net,
        element="load",
        multiplying_column="p_mw",
        relative_profiles=load_rel,
        profile_suffix="_pload",
        time_as_index=False,
    ),
    ("load", "q_mvar"): sb.get_absolute_profiles_from_relative_profiles(
        net,
        element="load",
        multiplying_column="q_mvar",
        relative_profiles=load_rel,
        profile_suffix="_qload",
        time_as_index=False,
    ),
    ("sgen", "p_mw"): sb.get_absolute_profiles_from_relative_profiles(
        net,
        element="sgen",
        multiplying_column="p_mw",
        relative_profiles=sgen_rel,
        time_as_index=False,
    ),
}

def apply_profile_step_cached(net, profiles, step):
    """Apply one cached global SimBench profile step using its original row index."""
    step = int(step)
    for (elm, param), df in profiles.items():
        if df.shape[1] and elm in net and len(net[elm]):
            net[elm].loc[df.columns, param] = df.loc[step].values

def run_pf_quiet(net):
    """Run Sarvesh's robust PF while suppressing expected solver spam for severe contingencies."""
    sink = io.StringIO()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        with redirect_stdout(sink), redirect_stderr(sink):
            return run_pf(net)

def apply_scenario_row_cached(net, profiles, row, sidx=None, battery_cfg=None):
    """Memory-safe scenario application using cached profile rows."""
    apply_profile_step_cached(net, profiles, int(row["profile_step"]))
    scale_injections(net, row["load_mult"], row["pv_mult"])

    if sidx is not None and battery_cfg is not None:
        net.storage.at[sidx, "soc_percent"] = row["battery_soc"] * 100

    info = None
    if row["outage"] in ("line", "trafo"):
        # Find the critical asset from the stressed-but-still-connected state first.
        if not run_pf_quiet(net):
            return ("pre_contingency_pf_failed",)

        if row["outage"] == "line":
            info = line_outage(net, pick_critical_line(net, "mv"))
        else:
            info = derate_transformer(net, pick_loaded_trafo(net), 0.6)

    return info

print(len(table), "scenarios |", table.scenario.nunique(), "types")
print("Cached profile steps:", len(needed_steps))
print("Load-P cache shape:", profiles[("load", "p_mw")].shape)
print("Load-Q cache shape:", profiles[("load", "q_mvar")].shape)
print("SGen-P cache shape:", profiles[("sgen", "p_mw")].shape)

# CRITICAL sanity check: cached annual peak must reproduce the already validated grid.
reset_contingencies(net, snap)
apply_profile_step_cached(net, profiles, 33001)
set_battery_power(net, sidx, 0.0)

assert run_pf_quiet(net), "Cached annual-peak profile does not converge."

peak_check = grid_metrics(net)
print("\nAnnual-peak cache validation:")
print("Vmin:", round(peak_check["v_min_pu"], 6))
print("Max line %:", round(peak_check["max_line_loading_pct"], 4))
print("Max trafo %:", round(peak_check["max_trafo_loading_pct"], 4))
print("Grid import MW:", round(peak_check["grid_import_mw"], 6))

# Expected values from the already validated battery/power-flow notebooks.
assert abs(peak_check["v_min_pu"] - 0.994604) < 0.002, "Peak Vmin mismatch"
assert abs(peak_check["max_line_loading_pct"] - 32.203874) < 1.0, "Peak line-loading mismatch"
assert abs(peak_check["max_trafo_loading_pct"] - 31.378085) < 1.0, "Peak trafo-loading mismatch"
assert abs(peak_check["grid_import_mw"] - 19.359202) < 1.0, "Peak import mismatch"

print("ANNUAL-PEAK CACHE VALIDATION: PASS")

display(
    table.groupby("scenario").first()[
        ["profile_step", "hour", "load_mult", "pv_mult",
         "price_mult", "battery_soc", "outage"]
    ]
)


## Run every scenario: battery OFF vs battery ON (simple support rule; PSO/fuzzy plug in via `battery_dispatch`)

In [ ]:
def support_rule(m, cfg):
    """Fallback support rule: discharge under grid stress; otherwise remain idle."""
    stressed = (
        m["v_min_pu"] < 0.97
        or m["max_line_loading_pct"] > 70
        or m["max_trafo_loading_pct"] > 70
    )
    return -cfg.p_max_mw if stressed else 0.0


records = []

for i, (_, row) in enumerate(table.iterrows(), start=1):
    reset_contingencies(net, snap)

    # Apply real SimBench state + multipliers + optional real contingency.
    info = apply_scenario_row_cached(net, profiles, row, sidx, cfg)

    set_battery_power(net, sidx, 0.0)
    ok_before = run_pf_quiet(net)

    if not ok_before:
        records.append(dict(
            scenario_id=row.scenario_id,
            scenario=row.scenario,
            profile_step=row.profile_step,
            contingency=str(info),
            battery_soc=row.battery_soc,
            battery_power_mw=np.nan,
            converged_before=False,
            converged_after=False,
        ))
        continue

    before = grid_metrics(net)

    E_now = row.battery_soc * cfg.e_cap_mwh
    p_cmd, E_next = battery_dispatch(
        support_rule(before, cfg),
        E_now,
        cfg,
    )
    set_battery_power(net, sidx, p_cmd)

    ok_after = run_pf_quiet(net)

    if not ok_after:
        rec = dict(
            scenario_id=row.scenario_id,
            scenario=row.scenario,
            profile_step=row.profile_step,
            contingency=str(info),
            battery_soc=row.battery_soc,
            battery_power_mw=p_cmd,
            converged_before=True,
            converged_after=False,
        )
        rec.update({f"{k}_before": v for k, v in before.items()})
        records.append(rec)
        continue

    after = grid_metrics(net)

    rec = dict(
        scenario_id=row.scenario_id,
        scenario=row.scenario,
        profile_step=row.profile_step,
        contingency=str(info),
        battery_soc=row.battery_soc,
        battery_power_mw=p_cmd,
        converged_before=True,
        converged_after=True,
    )
    rec.update({f"{k}_before": v for k, v in before.items()})
    rec.update({f"{k}_after": v for k, v in after.items()})
    records.append(rec)

    if i % 25 == 0:
        print(f"Completed {i}/{len(table)} scenarios")

res = pd.DataFrame(records)
res.to_csv("output/scenario_battery_results.csv", index=False)

reset_contingencies(net, snap)
set_battery_power(net, sidx, 0.0)

print("\nRows:", len(res))
print("Before PF failures:", int((~res["converged_before"]).sum()))
print("After PF failures:", int((~res["converged_after"]).sum()))

cols = [
    "v_min_pu",
    "max_line_loading_pct",
    "max_trafo_loading_pct",
    "grid_import_mw",
]

valid = res[res["converged_before"] & res["converged_after"]]
summary = valid.groupby("scenario")[
    [f"{c}_{s}" for c in cols for s in ("before", "after")]
].mean().round(3)

display(summary)


## Contingency demos (topology really changes)

In [ ]:
# Use a fresh network for contingency demos so no prior scenario state can leak in.
net_c = sb.get_simbench_net("1-MVLV-urban-all-0-sw")
cfg_c = BatteryConfig(**json.load(open("output/battery_config.json")))
sidx_c = add_battery(net_c, cfg_c)
snap_c = snapshot_state(net_c)

apply_profile_step_cached(net_c, profiles, 33001)
set_battery_power(net_c, sidx_c, 0.0)

assert run_pf_quiet(net_c), "Healthy annual-peak baseline power flow failed."
base = grid_metrics(net_c)
base["converged"] = True

print("Contingency baseline:")
print("Vmin:", round(base["v_min_pu"], 6),
      "| Max line %:", round(base["max_line_loading_pct"], 4),
      "| Max trafo %:", round(base["max_trafo_loading_pct"], 4),
      "| Import MW:", round(base["grid_import_mw"], 6))

# Pick all assets from the SAME healthy annual-peak state.
lid = pick_critical_line(net_c, "mv")
tid = pick_loaded_trafo(net_c)
gid = int(net_c.sgen["p_mw"].idxmax())

critical_line_switches = net_c.switch[
    (net_c.switch["et"] == "l")
    & (net_c.switch["closed"])
    & (net_c.switch["element"] == lid)
]

if len(critical_line_switches):
    sw = int(critical_line_switches.index[0])
else:
    sw = int(
        net_c.switch[
            (net_c.switch["et"] == "l") & (net_c.switch["closed"])
        ].index[0]
    )

tests = {}

def run_test(name, action):
    """Run each contingency independently from the same healthy annual-peak state."""
    reset_contingencies(net_c, snap_c)
    apply_profile_step_cached(net_c, profiles, 33001)
    set_battery_power(net_c, sidx_c, 0.0)

    action()
    ok = run_pf_quiet(net_c)

    if ok:
        metrics = grid_metrics(net_c)
        metrics["converged"] = True
    else:
        metrics = {
            "converged": False,
            "v_min_pu": np.nan,
            "max_line_loading_pct": np.nan,
            "max_trafo_loading_pct": np.nan,
            "grid_import_mw": np.nan,
            "unsupplied_buses": np.nan,
            "voltage_violation_count": np.nan,
            "overload_count": np.nan,
        }

    tests[name] = metrics

run_test(f"line {lid} out", lambda: line_outage(net_c, lid))
run_test(f"trafo {tid} derated 50%", lambda: derate_transformer(net_c, tid, 0.5))
run_test(f"sgen {gid} lost", lambda: remove_generator(net_c, gid))
run_test(f"switch {sw} opened", lambda: open_switch(net_c, sw))

reset_contingencies(net_c, snap_c)
apply_profile_step_cached(net_c, profiles, 33001)
set_battery_power(net_c, sidx_c, 0.0)

cont = pd.DataFrame({"BASE": base, **tests}).T

show_cols = [
    "converged",
    "v_min_pu",
    "max_line_loading_pct",
    "max_trafo_loading_pct",
    "grid_import_mw",
    "unsupplied_buses",
    "voltage_violation_count",
    "overload_count",
]

display(cont[show_cols].round(4))
cont.to_csv("output/contingency_results.csv")
